In [1]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('Hydro_to_Climate_Mapping.csv')

# Find the number of unique entries in the 'Hydro ID' column
unique_hydro_ids = df['Hydro ID'].nunique()

print(f"Number of unique entries in Hydro ID column: {unique_hydro_ids}")

Number of unique entries in Hydro ID column: 436


In [2]:
# Display the first 10 Hydro IDs
print("First 10 Hydro IDs:")
print(df['Hydro ID'].head(10).tolist())

First 10 Hydro IDs:
['08LG010', '08JA023', '08HB006', '08HA002', '08NA006', '08GD008', '08NL039', '08LB038', '08FE003', '08HD005']


In [4]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('Hydat.sqlite3')

# This pulls just the first 5 rows so you can see the actual data
df_sample = pd.read_sql_query("SELECT * FROM STATIONS LIMIT 5;", conn)

# Display column names
print(df_sample.columns.tolist())

print('########################################')

# Or see the first few rows to verify the data
print(df_sample[['STATION_NUMBER', 'STATION_NAME', 'DRAINAGE_AREA_GROSS']])

conn.close()

['STATION_NUMBER', 'STATION_NAME', 'PROV_TERR_STATE_LOC', 'REGIONAL_OFFICE_ID', 'HYD_STATUS', 'SED_STATUS', 'LATITUDE', 'LONGITUDE', 'DRAINAGE_AREA_GROSS', 'DRAINAGE_AREA_EFFECT', 'RHBN', 'REAL_TIME', 'CONTRIBUTOR_ID', 'OPERATOR_ID', 'DATUM_ID']
########################################
  STATION_NUMBER                                       STATION_NAME  \
0        01AA002    DAAQUAM (RIVIERE) EN AVAL DE LA RIVIERE SHIDGEL   
1        01AD001  MADAWASKA (RIVIERE) A 6 KM EN AVAL DU BARRAGE ...   
2        01AD002                      SAINT JOHN RIVER AT FORT KENT   
3        01AD003        ST. FRANCIS RIVER AT OUTLET OF GLASIER LAKE   
4        01AD004                     SAINT JOHN RIVER AT EDMUNDSTON   

   DRAINAGE_AREA_GROSS  
0                598.0  
1               2690.0  
2              14700.0  
3               1350.0  
4              15500.0  


In [ ]:
import sqlite3
import pandas as pd

mapping_file_path = 'Hydro_to_Climate_Mapping.csv'
hydat_db_path = 'Hydat.sqlite3'
output_file_path = 'updated_hydro_stations.csv'

df_mapping = pd.read_csv(mapping_file_path, index_col=0)

# Connect to HYDAT -  pulling ONLY Gross Area
conn = sqlite3.connect(hydat_db_path)
query = """
SELECT 
    STATION_NUMBER, 
    DRAINAGE_AREA_GROSS 
FROM STATIONS
"""
df_hydat = pd.read_sql_query(query, conn)
conn.close()

# Join drainage area to original dataframe
df_final = pd.merge(
    df_mapping, 
    df_hydat, 
    left_on='Hydro ID', 
    right_on='STATION_NUMBER', 
    how='left'
).drop(columns=['STATION_NUMBER'])

missing_count = df_final['DRAINAGE_AREA_GROSS'].isna().sum()
print(f"Merge Complete.")
print(f"Total Stations: {len(df_final)}")
print(f"Stations missing Gross Area: {missing_count}")

df_final.to_csv(output_file_path, index=False)

Merge Complete.
Total Stations: 436
Stations missing Gross Area: 67
